# StarEmbed: Benchmarking Time Series Foundation Models on Astronomical Observations of Variable Stars

This [paper](https://arxiv.org/abs/2510.06200) make a benchmark on ZTF dataset in order to evaluate foundatoinal models such as Astromer or Chronos

## Proposal

I will test whether the released StarEmbed pipeline reproduces the clustering performance reported for frozen Chronos-tiny embeddings on ZTF variable stars. Our target is the entry “0.1596(0.0029)” in Table 2, row “Chronos”, column “ARI (K-Means)”. I will rerun the experiment across multiple random seeds and examine how the K-means initialization procedure affects the result.

## Load data

Load dataset ZTF with train/val/test splits from huggingface

In [11]:
import os
from datasets import load_dataset, load_from_disk
from huggingface_hub import login, whoami

In [12]:
os.environ["HF_HUB_DISABLE_XET"] = "1"
hf_token = os.getenv("HF_TOKEN")

In [13]:
login(token=hf_token)

user = whoami()
print(user['name'])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


astronaut-miem-2024


In [14]:
output_dir = "./datasets/ZTF"
dataset = load_dataset("123anonymous123/StarEmbed")

for split_name, split_data in dataset.items():
    # Map split names to desired folder names
    folder_name = {
        "train": "train",
        "test": "test",
        "validation": "validation",
    }.get(split_name, split_name)

    split_path = os.path.join(output_dir, folder_name)
    split_data.save_to_disk(split_path)
    print(f"Saved {split_name} ({len(split_data)} rows) to {split_path}")

Saving the dataset (2/2 shards): 100%|██████████| 29047/29047 [00:00<00:00, 29830.81 examples/s]


Saved train (29047 rows) to ./datasets/ZTF/train


Saving the dataset (1/1 shards): 100%|██████████| 4150/4150 [00:00<00:00, 7590.27 examples/s]


Saved validation (4150 rows) to ./datasets/ZTF/validation


Saving the dataset (1/1 shards): 100%|██████████| 8301/8301 [00:00<00:00, 30131.55 examples/s]

Saved test (8301 rows) to ./datasets/ZTF/test


In [ ]:
val = load_from_disk("./datasets/ZTF/validation")
val = val.with_format(None)

print("Number of light curves:", len(val))
print("Columns:", val.column_names)
print("Classes:", val.unique("class_str"))

Количество звёзд: 4150
Колонки: ['sourceid', 'bands_data', 'item_id', 'start', 'freq', 'period', 'objid', 'csdr1_id', 'class_str']
Классы: ['1', '2', '4', '5', '8', '6', '13']


## Smoke test
> for November 2th

Select data for smoke test:
- 10 objects per class
- star has `g` and `r` band
- possible to normalize the light curve
- must be >= 32 point per curve

In [16]:
import numpy as np
from collections import Counter


def usable_curve(curve):
    if curve is None:
        return False

    time = np.asarray(curve["mjd"], dtype=float)
    mag = np.asarray(curve["target"], dtype=float)

    return (
        len(time) == len(mag)
        and len(mag) >= 2
        and np.isfinite(time).all()
        and np.isfinite(mag).all()
        and mag.std() > 0
    )


classes = val.unique("class_str")
indices = []
seen_ids = set()
counts = Counter()

for index, star in enumerate(val):
    label = star["class_str"]
    source_id = star["sourceid"]

    if counts[label] == 10 or source_id in seen_ids:
        continue

    g = star["bands_data"]["g"]
    r = star["bands_data"]["r"]

    if not usable_curve(g) or not usable_curve(r):
        continue
    if len(g["target"]) + len(r["target"]) < 32:
        continue

    indices.append(index)
    seen_ids.add(source_id)
    counts[label] += 1

    if len(indices) == 70:
        break

print("Classes:", {label: counts[label] for label in classes})

smoke = val.select(indices)

Classes: {'1': 10, '2': 10, '4': 10, '5': 10, '8': 10, '6': 10, '13': 10}


Let check the data

In [17]:
star = smoke[0]

print("ID of the star:", star["sourceid"])
print("Class:", star["class_str"])

for band in ("g", "r"):
    curve = star["bands_data"][band]
    time = curve["mjd"]
    mag = curve["target"]

    print(f"\nBand {band}: {len(mag)} observations")
    print("First 5 points (time (mjd), magnitude):")
    print(list(zip(time[:5], mag[:5])))

ID of the star: 131863309293204828
Class: 1

Band g: 432 observations
First 5 points (time (mjd), magnitude):
[(58234.5, 15.79605484008789), (58247.48046875, 15.903013229370117), (58254.4453125, 15.850971221923828), (58262.46484375, 16.438880920410156), (58269.453125, 16.262901306152344)]

Band r: 590 observations
First 5 points (time (mjd), magnitude):
[(58244.48046875, 15.673005104064941), (58247.47265625, 15.082261085510254), (58254.48046875, 15.075263977050781), (58257.421875, 15.392078399658203), (58257.4609375, 15.250353813171387)]


Save smoke data as dataset

In [18]:
smoke.save_to_disk("./datasets/ZTF/ztf_smoke70")

Saving the dataset (1/1 shards): 100%|██████████| 70/70 [00:00<00:00, 1372.90 examples/s]


In the terminal I launch commands in order to get embeddings for light curves with Chronos model. For that I use this commands:
- for Chronos-t5
```bash
python embed.py --ctx 200 --model t5
```
- for Chronos-bolt
```bash
python embed.py --ctx 200 --model bolt
```

With embeddings after that I compute embeddings with concat for band g and r as in the paper
```bash
python compute_avg_embeddings.py \
  --base_path /Users/raimova-al/Documents/study/master/research_seminar/datasets \    
  --dataset ZTF \ 
  --splits ztf_smoke70_t5 ztf_smoke70_bolt \
  --bands g r \
  --band_combination concat \
  --batch_size 8
```

Below, I reimplement same code for clustering as in the [clustering.py](https://github.com/skai-institute/StarEmbed/blob/41f6220d9e37203543745cd6c110f1f9ad5635cc/src/benchmark/clustering/clustering.py) because there is some limitations in input folder. With this I get metrics for clustering

In [22]:
import logging
import os
import pathlib
import random

import matplotlib.pyplot as plt
import numpy as np
from datasets import DatasetDict, load_from_disk
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from sklearn.metrics import (
    adjusted_rand_score,
    confusion_matrix,
    f1_score,
    normalized_mutual_info_score,
)
from sklearn.preprocessing import StandardScaler


def _clustering_metrics(y_true, y_pred):
    """ARI, NMI and macro-F1. Cluster ids are aligned with the true labels before F1."""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    ari = adjusted_rand_score(y_true, y_pred)
    nmi = normalized_mutual_info_score(y_true, y_pred)
    labels = np.unique(np.concatenate([y_true, y_pred]))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    row_ind, col_ind = linear_sum_assignment(-cm)
    mapping = {labels[col]: labels[row] for row, col in zip(row_ind, col_ind)}
    y_mapped = np.array([mapping[label] for label in y_pred])
    f1 = f1_score(y_true, y_mapped, average="macro")
    return ari, nmi, f1


def _load_any(dataset_dir):
    """Load a Dataset, a DatasetDict, or a folder that contains split subfolders."""
    dataset_dir = os.path.abspath(dataset_dir)
    if os.path.isfile(os.path.join(dataset_dir, "dataset_info.json")) or os.path.isfile(
        os.path.join(dataset_dir, "dataset_dict.json")
    ):
        return load_from_disk(dataset_dir)

    split_names = [
        name
        for name in ("train", "validation", "test")
        if os.path.isfile(os.path.join(dataset_dir, name, "dataset_info.json"))
    ]
    if not split_names:
        raise FileNotFoundError(
            f"{dataset_dir} is not a Hugging Face dataset and has no train/validation/test folders"
        )
    return DatasetDict({name: load_from_disk(os.path.join(dataset_dir, name)) for name in split_names})


def _remove_outliers(dataset, hand_crafted):
    """Drop the same rows as clustering.py when those indices exist in this dataset."""
    needed = ("train", "validation", "test")
    if not isinstance(dataset, DatasetDict) or any(name not in dataset for name in needed):
        print("Skipping outlier removal: input is not a train/validation/test DatasetDict")
        return dataset

    if not hand_crafted:
        bad = {"train": [23082], "validation": [473], "test": [7880]}
        print("Removing outliers from time series embedding dataset")
    else:
        bad = {"train": [3010, 9693, 16524, 22151], "validation": [449], "test": [1158]}
        print("Removing outliers from hand-crafted feature dataset")

    if any(index >= len(dataset[split]) for split, indices in bad.items() for index in indices):
        print("Skipping outlier removal: the known outlier indices are outside this dataset")
        return dataset

    for split, indices in bad.items():
        keep = [i for i in range(len(dataset[split])) if i not in set(indices)]
        dataset[split] = dataset[split].select(keep)
    print(
        f"Selected {len(dataset['train'])} train, "
        f"{len(dataset['validation'])} validation, {len(dataset['test'])} test samples"
    )
    return dataset


def _embedding_matrix(values):
    matrix = np.asarray(values, dtype=np.float32)
    if matrix.ndim == 1:
        matrix = np.stack([np.asarray(row, dtype=np.float32) for row in values])
    return matrix


def _class_mask(labels, classes):
    if str(classes).lower() == "all":
        return np.ones(len(labels), dtype=bool)
    wanted = set()
    for token in str(classes).split(","):
        token = token.strip()
        if not token:
            continue
        wanted.add(token)
        try:
            wanted.add(int(token))
            wanted.add(str(int(token)))
        except ValueError:
            pass
    return np.array([label in wanted or str(label) in wanted for label in labels])


def _save_dendrogram(linkage_matrix, n_clusters, split_name, save_dir):
    plt.figure(figsize=(12, 8))
    dendrogram(
        linkage_matrix,
        truncate_mode="lastp",
        p=30,
        show_leaf_counts=True,
        leaf_rotation=90,
    )
    distances = sorted(set(linkage_matrix[:, 2]))
    for dist in reversed(distances):
        if len(set(fcluster(linkage_matrix, t=dist, criterion="distance"))) == n_clusters:
            plt.axhline(
                y=dist,
                color="red",
                linestyle="--",
                linewidth=2,
                label=f"Cut for {n_clusters} clusters",
            )
            break
    plt.title(f"Hierarchical Clustering Dendrogram - {split_name}")
    plt.xlabel("Cluster Size (or Index)")
    plt.ylabel("Distance")
    plt.legend()
    dendrogram_file = os.path.join(save_dir, f"dendrogram_{split_name.lower().replace(' ', '_')}.pdf")
    plt.savefig(dendrogram_file, bbox_inches="tight", dpi=300)
    plt.close()
    print(f"  → Dendrogram saved to {dendrogram_file}")
    return dendrogram_file


def _plot_tsne(features, labels, title, perplexity, seed, save_dir):
    n_samples = len(features)
    if n_samples < 2:
        print(f"  → t-SNE skipped for {title}: fewer than 2 samples")
        return None
    used_perplexity = perplexity
    if used_perplexity >= n_samples:
        used_perplexity = max(1.0, n_samples - 1)
        print(f"  → t-SNE perplexity lowered from {perplexity} to {used_perplexity}")
    print(f"  → t-SNE ({title})")
    embedded = TSNE(n_components=2, perplexity=used_perplexity, random_state=seed).fit_transform(features)
    plt.figure(figsize=(7, 6))
    for label in np.unique(labels):
        mask = labels == label
        plt.scatter(embedded[mask, 0], embedded[mask, 1], s=30, label=str(label), alpha=0.7)
    plt.legend(title="Class", bbox_to_anchor=(1.02, 1))
    plt.title(title)
    plt.axis("off")
    plt.tight_layout()
    plot_path = os.path.join(save_dir, f"{'-'.join(title.split(' '))}.pdf")
    print(f"Saving t-SNE plot to {plot_path}")
    plt.savefig(plot_path)
    plt.close()
    return plot_path


def _cluster_split(features, label_idx, n_clusters, seed, method, save_dendrogram, save_dir, split_name):
    kmeans_metrics = None
    ward_metrics = None
    if method in ("kmeans", "both"):
        print(f"  → K-Means (k={n_clusters})")
        model = KMeans(n_clusters=n_clusters, random_state=seed).fit(features)
        kmeans_metrics = _clustering_metrics(label_idx, model.labels_)
    if method in ("hierarchical", "both"):
        print(f"  → Ward    (k={n_clusters})")
        linkage_matrix = linkage(features, method="ward")
        clusters = fcluster(linkage_matrix, t=n_clusters, criterion="maxclust")
        ward_metrics = _clustering_metrics(label_idx, clusters)
        if save_dendrogram and save_dir:
            _save_dendrogram(linkage_matrix, n_clusters, split_name, save_dir)
    return kmeans_metrics, ward_metrics


def run_clustering_pipeline(
    dataset_dir,
    output_dir=None,
    perplexity=30,
    random_state=42,
    mode="test",
    classes="all",
    scenario="concat",
    hand_crafted=0,
    standardize=0,
    clustering_method="both",
    save_dendrogram=False,
):
    """Run the StarEmbed clustering pipeline and write the same artifacts.

    ``dataset_dir`` may be a single Hugging Face dataset, a DatasetDict, or a
    directory with ``train``, ``validation`` and ``test`` subfolders. A single
    dataset is clustered as one split. The result directory contains ``log.txt``,
    a t-SNE PDF, and a dendrogram PDF when ``save_dendrogram`` is set.
    """
    if mode not in ("train", "test", "validation", "both", "all"):
        raise ValueError("mode must be train, test, validation, both, or all")
    if clustering_method not in ("kmeans", "hierarchical", "both"):
        raise ValueError("clustering_method must be kmeans, hierarchical, or both")

    random.seed(random_state)
    np.random.seed(random_state)

    dataset_dir = os.path.abspath(dataset_dir)
    if output_dir is None:
        output_dir = os.path.join(os.path.dirname(dataset_dir), "clustering_out")
    experiment_name = (
        f"{pathlib.Path(dataset_dir).name}_{mode}_{scenario}_std{int(bool(standardize))}"
        f"_p{perplexity}_seed{random_state}"
    )
    result_dir = os.path.join(output_dir, experiment_name)
    pathlib.Path(result_dir).mkdir(parents=True, exist_ok=True)

    logger = logging.getLogger("starembed.clustering")
    logger.handlers.clear()
    logger.setLevel(logging.INFO)
    logger.propagate = False
    handler = logging.FileHandler(os.path.join(result_dir, "log.txt"), mode="w")
    handler.setFormatter(logging.Formatter("INFO:__main__:%(message)s"))
    logger.addHandler(handler)

    def emit(message):
        print(message)
        logger.info(message)

    artifacts = {
        "all": ("All   ", "All Splits", "t-SNE on Combined Dataset", "Step 5: Clustering ALL splits combined (train+validation+test)", "Step 9: t-SNE ALL splits combined (original labels)"),
        "train": ("Train ", "Train Split", "t-SNE on Filtered Train", "Step 6: Clustering TRAIN split", "Step 10: t-SNE TRAIN (original labels)"),
        "validation": ("Valid ", "Validation Split", "t-SNE on Filtered Validation", "Step 7: Clustering VALIDATION split", "Step 11: t-SNE VALIDATION (original labels)"),
        "test": ("Test  ", "Test Split", "t-SNE on Filtered Test", "Step 8: Clustering TEST split", "Step 12: t-SNE TEST (original labels)"),
    }

    try:
        emit(f"Saving results to {result_dir}")
        emit("Step 1: Loading dataset...")
        dataset = _load_any(dataset_dir)
        dataset = _remove_outliers(dataset, hand_crafted=bool(hand_crafted))
        print("Using pre-computed combined_embedding")

        if isinstance(dataset, DatasetDict):
            split_names = [name for name in ("train", "validation", "test") if name in dataset]
            if not split_names:
                split_names = list(dataset.keys())
            arrays = {name: dataset[name] for name in split_names}
        else:
            arrays = {"__single__": dataset}

        prepared = {}
        for name, split in arrays.items():
            if "combined_embedding" not in split.column_names:
                raise ValueError(
                    f"{name} has no combined_embedding column. Run compute_avg_embeddings.py first."
                )
            features = _embedding_matrix(split["combined_embedding"])
            original = np.asarray(split["class_str"])
            mask = _class_mask(original, classes)
            prepared[name] = (features[mask], original[mask])

        reference_name = "train" if "train" in prepared else next(iter(prepared))
        reference_labels = prepared[reference_name][1]
        if len(reference_labels) == 0:
            raise ValueError("No rows left after class filtering")
        label_order = sorted(set(reference_labels.tolist()), key=str)
        label_to_index = {label: index for index, label in enumerate(label_order)}
        emit("Step 2: Building label2idx mapping and saving original labels...")
        emit(f"Automatic label order: {label_order}")
        emit("Step 3: Converting embeddings & labels to NumPy arrays...")
        if str(classes).lower() == "all":
            emit("Step 4: No filtering (using all classes)")
        else:
            emit(f"Step 4: Filtering to classes {classes}...")

        indexed = {}
        for name, (features, original) in prepared.items():
            missing = [label for label in original if label not in label_to_index]
            if missing:
                raise ValueError(f"{name} contains labels that are absent from {reference_name}: {set(map(str, missing))}")
            indexed[name] = (features, np.array([label_to_index[label] for label in original]), original)

        n_clusters = len(np.unique(indexed[reference_name][1]))
        if n_clusters < 1:
            raise ValueError("Need at least one class to cluster")

        if standardize:
            emit("Step 4.5: Applying StandardScaler to features...")
            scaler = StandardScaler()
            if "train" in indexed:
                train_features, train_idx, train_orig = indexed["train"]
                indexed["train"] = (scaler.fit_transform(train_features), train_idx, train_orig)
                for name in indexed:
                    if name == "train":
                        continue
                    features, label_idx, original = indexed[name]
                    indexed[name] = (scaler.transform(features), label_idx, original)
            else:
                for name, (features, label_idx, original) in indexed.items():
                    indexed[name] = (scaler.fit_transform(features), label_idx, original)
            shape_report = ", ".join(
                f"{'dataset' if name == '__single__' else name} shape: {indexed[name][0].shape}"
                for name in indexed
            )
            emit(f"Features standardized. {shape_report}")
        else:
            emit("Step 4.5: Skipping standardization")

        single_dataset = "__single__" in indexed
        if single_dataset:
            emit("Step 4.6: Using the whole dataset")
            targets = ["all" if mode in ("all", "both") else mode]
        elif mode == "all":
            emit("Step 4.6: Combining all splits (train+validation+test) for clustering...")
            order = [name for name in ("train", "validation", "test") if name in indexed]
            indexed["all"] = (
                np.vstack([indexed[name][0] for name in order]),
                np.concatenate([indexed[name][1] for name in order]),
                np.concatenate([indexed[name][2] for name in order]),
            )
            emit(f"Combined dataset shape: {indexed['all'][0].shape}")
            targets = ["all"]
        elif mode == "both":
            emit("Step 4.6: Using individual splits")
            targets = [name for name in ("train", "test") if name in indexed]
        else:
            emit("Step 4.6: Using individual splits")
            if mode not in indexed:
                raise ValueError(f"mode={mode!r}, but the dataset only has splits {list(indexed)}")
            targets = [mode]

        if single_dataset and targets == ["all"]:
            indexed["all"] = indexed.pop("__single__")
        elif single_dataset:
            indexed[targets[0]] = indexed.pop("__single__")

        for target in targets:
            prefix, split_name, title, cluster_step, tsne_step = artifacts[target]
            features, label_idx, original = indexed[target]
            if len(features) < n_clusters:
                raise ValueError(
                    f"{target} has {len(features)} rows, which is fewer than {n_clusters} clusters"
                )
            print()
            emit(cluster_step)
            kmeans_metrics, ward_metrics = _cluster_split(
                features,
                label_idx,
                n_clusters,
                random_state,
                clustering_method,
                save_dendrogram,
                result_dir,
                split_name,
            )
            if kmeans_metrics:
                ari, nmi, f1 = kmeans_metrics
                emit(f"{prefix}→ K-Means  ARI={ari:.4f}, NMI={nmi:.4f}, F1={f1:.4f}")
            if ward_metrics:
                ari, nmi, f1 = ward_metrics
                emit(f"{prefix}→ Ward      ARI={ari:.4f}, NMI={nmi:.4f}, F1={f1:.4f}")
            print()
            emit(tsne_step)
            _plot_tsne(features, original, title, perplexity, random_state, result_dir)

        emit("Clustering pipeline completed successfully.")
        return result_dir
    except Exception as error:
        logger.error(f"Error: {error}")
        logger.exception("Error")
        raise
    finally:
        handler.close()
        logger.removeHandler(handler)


Metrics for t5 Chronos model

In [23]:
result_dir = run_clustering_pipeline(
    "/Users/raimova-al/Documents/study/master/research_seminar/datasets/ZTF/ztf_smoke70_chronos_t5_ctx200_bandgr",
    mode="all",
    scenario="concat",
    clustering_method="both",
    standardize=1,
    perplexity=30,
    random_state=42,
    save_dendrogram=True,
)
print(result_dir)

Saving results to /Users/raimova-al/Documents/study/master/research_seminar/datasets/ZTF/clustering_out/ztf_smoke70_chronos_t5_ctx200_bandgr_all_concat_std1_p30_seed42
Step 1: Loading dataset...
Skipping outlier removal: input is not a train/validation/test DatasetDict
Using pre-computed combined_embedding
Step 2: Building label2idx mapping and saving original labels...
Automatic label order: ['1', '13', '2', '4', '5', '6', '8']
Step 3: Converting embeddings & labels to NumPy arrays...
Step 4: No filtering (using all classes)
Step 4.5: Applying StandardScaler to features...
Features standardized. dataset shape: (70, 512)
Step 4.6: Using the whole dataset

Step 5: Clustering ALL splits combined (train+validation+test)
  → K-Means (k=7)
  → Ward    (k=7)
  → Dendrogram saved to /Users/raimova-al/Documents/study/master/research_seminar/datasets/ZTF/clustering_out/ztf_smoke70_chronos_t5_ctx200_bandgr_all_concat_std1_p30_seed42/dendrogram_all_splits.pdf
All   → K-Means  ARI=0.2177, NMI=0.40

Metrics for bolt Chronos model

In [24]:
run_clustering_pipeline(
    "/Users/raimova-al/Documents/study/master/research_seminar/datasets/ZTF/ztf_smoke70_chronos_bolt_ctx200_bandgr",
    mode="all",
    scenario="concat",
    clustering_method="both",
    standardize=1,
    perplexity=30,
    random_state=42,
    save_dendrogram=True,
)

Saving results to /Users/raimova-al/Documents/study/master/research_seminar/datasets/ZTF/clustering_out/ztf_smoke70_chronos_bolt_ctx200_bandgr_all_concat_std1_p30_seed42
Step 1: Loading dataset...
Skipping outlier removal: input is not a train/validation/test DatasetDict
Using pre-computed combined_embedding
Step 2: Building label2idx mapping and saving original labels...
Automatic label order: ['1', '13', '2', '4', '5', '6', '8']
Step 3: Converting embeddings & labels to NumPy arrays...
Step 4: No filtering (using all classes)
Step 4.5: Applying StandardScaler to features...
Features standardized. dataset shape: (70, 512)
Step 4.6: Using the whole dataset

Step 5: Clustering ALL splits combined (train+validation+test)
  → K-Means (k=7)
  → Ward    (k=7)
  → Dendrogram saved to /Users/raimova-al/Documents/study/master/research_seminar/datasets/ZTF/clustering_out/ztf_smoke70_chronos_bolt_ctx200_bandgr_all_concat_std1_p30_seed42/dendrogram_all_splits.pdf
All   → K-Means  ARI=0.1881, NMI=

'/Users/raimova-al/Documents/study/master/research_seminar/datasets/ZTF/clustering_out/ztf_smoke70_chronos_bolt_ctx200_bandgr_all_concat_std1_p30_seed42'